# Metrics for many classes & many labels

_Metrics & Evaluation — measuring models, data & statistics_

**How to score a model when there are many classes — or when each example can wear several labels at once.**

With two classes you had one confusion matrix and one precision/recall pair. With many classes you get a bigger confusion matrix and a precision/recall pair per class — then you must summarize them into one number (that is what macro / micro / weighted do).

       With multilabel the very notion of "correct" changes: an answer is now a set of labels, so we measure how much two sets overlap (Hamming loss, example-based F1) or how well the model ranks the true labels to the top (LRAP, coverage, ranking loss).

---

This notebook is a practice scaffold for the **Metrics for many classes & many labels** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (top_k_accuracy_score, classification_report,
    hamming_loss, accuracy_score, f1_score,
    label_ranking_average_precision_score, coverage_error, label_ranking_loss)

# ---- MULTICLASS: 10-class handwritten digits ----
X, y = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=5000, random_state=0).fit(Xtr, ytr)

pred  = clf.predict(Xte)
proba = clf.predict_proba(Xte)          # scores for top-k

print("top-1 accuracy:", round(top_k_accuracy_score(yte, proba, k=1), 4))
print("top-3 accuracy:", round(top_k_accuracy_score(yte, proba, k=3), 4))
# per-class precision/recall/F1 + macro / micro / weighted summary rows:
print(classification_report(yte, pred, digits=3))

# ---- MULTILABEL: 5 examples, 3 labels each ----
Y_true = np.array([[1,0,1],[0,1,1],[1,1,0],[0,0,1],[1,0,0]])
Y_pred = np.array([[1,0,1],[0,1,0],[1,1,1],[0,0,1],[0,0,0]])
# model scores (confidences) used by the ranking metrics:
Y_score = np.array([[0.7,0.9,0.2],[0.3,0.6,0.5],[0.6,0.5,0.7],
                    [0.1,0.2,0.7],[0.4,0.5,0.2]])

print("hamming loss       :", round(hamming_loss(Y_true, Y_pred), 4))
print("subset/exact match :", round(accuracy_score(Y_true, Y_pred), 4))
print("F1 micro           :", round(f1_score(Y_true, Y_pred, average="micro"), 4))
print("F1 macro           :", round(f1_score(Y_true, Y_pred, average="macro"), 4))
print("F1 samples (ex-based):", round(f1_score(Y_true, Y_pred, average="samples"), 4))
print("F1 weighted        :", round(f1_score(Y_true, Y_pred, average="weighted"), 4))
print("LRAP               :", round(label_ranking_average_precision_score(Y_true, Y_score), 4))
print("coverage error     :", round(coverage_error(Y_true, Y_score), 4))
print("label ranking loss :", round(label_ranking_loss(Y_true, Y_score), 4))

## Visualize the data & results

_On real 10-class digit data, which class does the model handle worst — and how much does that hide inside a single overall score?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X, y = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=5000, random_state=0).fit(Xtr, ytr)
pred = clf.predict(Xte)

per_class = f1_score(yte, pred, average=None)   # one F1 per digit 0..9
worst = int(np.argmin(per_class))               # -> 8
print("per-class F1:", np.round(per_class, 3))
print("worst class:", worst, "F1=", round(per_class[worst], 3))
print("macro F1   :", round(f1_score(yte, pred, average="macro"), 3))
print("micro F1   :", round(f1_score(yte, pred, average="micro"), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A 10-class digit model gets micro-F1 = 0.96 but its per-class F1 for the digit '8' is only 0.92. The leaderboard shows micro-F1. Why might that be hiding a problem, and what should you report?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what micro does: it pools every example's right/wrong counts across all 10 classes into one tally. — _Because big and easy classes contribute most of the pooled counts, a weak class barely moves the micro number._
- Compute the macro-F1 (equal-weight mean of the 10 per-class F1) and read the per-class row. — _Macro gives class '8' a full 1/10 vote, so its weakness shows; the per-class row pinpoints exactly where the model fails._

**Answer:** Micro-F1 = 0.96 is dominated by the many easy digits, so the weak '8' (F1 = 0.92) is invisible in it. Report macro-F1 and the full per-class F1 table; the lowest per-class score (here '8') is the real risk to watch.

</details>

**Problem 2.** In a multilabel tagging task, a model is 90% correct on each individual label, yet its subset (exact-match) accuracy is only ~40%. Is the model broken?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note subset accuracy needs the ENTIRE predicted set to equal the true set for an example to count. — _One wrong label anywhere on an example zeroes the whole row, no partial credit._
- Estimate: if there are ~5 labels and each is right 90% of the time, the chance ALL five are right is about $0.9^5\approx0.59$ — and harder cases push it lower. — _Per-label accuracy and exact-match measure different things; high per-label accuracy can still give low exact-match._

**Answer:** Not necessarily. Exact-match is brutally strict, so strong per-label accuracy still yields modest subset accuracy. Report Hamming loss (here ~0.10) and example-based F1 alongside it for a fair picture.

</details>

**Problem 3.** For a label-ranking model you get LRAP = 0.73, coverage error = 2.2, and label ranking loss = 0.5. In plain words, what do these say?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read LRAP: average over true labels of (fraction of labels ranked at or above it that are also true). 1.0 = all true labels sit at the very top. — _0.73 means true labels are usually near the top but some wrong labels slip above them._
- Read coverage error and ranking loss: how far down to gather all true labels, and the fraction of (true, false) label pairs that are mis-ordered. — _Coverage 2.2 = scan about 2.2 labels deep; ranking loss 0.5 = half the right/wrong pairs are in the wrong order._

**Answer:** True labels usually rank high (LRAP 0.73) but not perfectly; you must scan ~2.2 labels down to catch all true ones (coverage 2.2), and half of the true-vs-false label pairs are mis-ordered (ranking loss 0.5). Lower coverage and ranking loss, higher LRAP, are better.

</details>